[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/lie_advection/lie_advection.ipynb)

# Discrete Lie Advection

This example simulates Lie advection of magnetic flux across a fixed planar triangulation of a swirling conductor. In a perfectly conducting fluid, magnetic flux is frozen into the moving medium and carried along with the flow. As a conceptual starting point, a single moving triangle shows how each edge sweeps a ribbon through spacetime so that the rate of change of its circulation combines endpoint potential readings with the flux across its swept ribbon, while around a closed loop the endpoint terms cancel pairwise. Across a swirling disk, differential rotation winds two lobes of opposite magnetic flux into thin spiral layers, whereas electrical resistance smooths the layers and dissipates magnetic energy without altering the total flux.

In 2+1D spacetime, the electromagnetic field is an extrinsic bivector `gradients ^ face_potentials` whose spatial plane carries magnetic flux and whose planes through time carry electric fields. An edge moving at `velocity` sweeps the intrinsic spacetime patch `(time + velocity) & edge`, and pairing the field directly with that world sheet `face_field & world_sheets` gives the magnetic flux rate with no metric. Ideal Ohm's law zeroes the field on comoving world sheets, so a fixed edge's reading changes by the magnetic flux across `fixed_sheets - comoving_sheets`, while pulling the vacuum duality map back through the Whitney curl `curl.adjugate()(vacuum(curl))` drives resistive diffusion on interior edges. Heumann and Hiptmair develop the related extrusion-contraction discretization in [*Extrusion Contraction Upwind Schemes for Convection-Diffusion Problems on Differential Forms*](https://sam.math.ethz.ch/sam_reports/reports_final/reports2011/2011-17.pdf).

In Gibbs–Heaviside vector notation, magnetic transport in a conductor moving at velocity $\mathbf v$ with resistivity $\eta$ splits into double cross products for induction and resistive curl-curl diffusion, $\partial_t \mathbf B = \nabla \times (\mathbf v \times \mathbf B) - \nabla \times (\eta\,\nabla \times \mathbf B)$, with $\mathbf B = \nabla \times \mathbf A$ tracked as an axial vector perpendicular to the plane. In differential-form and finite element exterior calculus notation, transport along $v$ splits through Cartan's identity $\mathcal L_v A = d(\iota_v A) + \iota_v(dA)$, while the discrete curl-curl operator assembles a primal-dual Hodge mass matrix $M_2 = \operatorname{diag}(1 / |f|)$ inside $S_1 = d_1^\top M_2 d_1$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import STA2D as ga
from numga.sparse import spdiag
from numga.topology import Topology
from examples.electromagnetism.lie_advection import render
from examples.util import shapes
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Plane = ga.gatype.vector()                                                   # Vector+
Upright = Plane.derive.within(Algebra("x+y+"))                               # Vector+
Field = ga.gatype.bivector()                                                 # Bivector+
# The magnetic field, a spatial bivector plane.
Magnetic = Field.derive.within(Algebra("x+y+"))                              # Bivector+
Point = ga.gatype.antivector()                                               # Antivector-
Patch = ga.gatype.antibivector()                                             # Antibivector-
Antiscalar = ga.gatype.antiscalar()                                          # Antiscalar-
ScalarMap = ga.gatype((Scalar, Scalar))                                      # Scalar+ <- Scalar+
CurlMap = ga.gatype((Magnetic, Scalar))                                      # Magnetic+ <- Scalar+
StiffnessMap = ga.gatype((Antiscalar, Scalar))                               # Antiscalar- <- Scalar+
triangle = Topology(3, [[0, 1, 2]])

# The unit future direction of an observer at rest in the spatial frame, pairing to 1 with `mv.t`.
time = -mv.t.dual()                                                          # [] Point-

np.set_printoptions(precision=4, suppress=True)

## Swept flux and endpoint readings

Along a moving edge, the circulation reading changes both at its endpoints, where the potential pairs with each corner's world line, and across the world sheet the edge sweeps through spacetime. Around a closed triangle, each corner belongs to one incoming and one outgoing edge, so the endpoint readings cancel pairwise and only the flux through the three swept boundary sheets remains.

In [ ]:
from examples.electromagnetism.lie_advection.scenarios import (
    corner_potentials, corner_velocities, ribbon_step, triangle_corners,
)

In [ ]:
def cartan_decomposition(corners: Point, corner_potentials: Plane,
                         corner_velocities: Point) -> tuple[Scalar, Scalar, Patch, Field]:
    """A moving triangle's edge readings change by endpoint potential differences (`[3] Scalar+[1]`) plus world-sheet fluxes (`[3] Scalar+[1]`) from the world sheets (`[3] Patch-[1]`) and face field (`Field+[1]`)."""
    sides = triangle.boundary[2, 1] * (triangle.d[0] * corners)              # [3] Point-[1]
    patch = (sides[0] & sides[1]) / 2                                        # Patch-[1]
    # Each vertex function is constant in time, so its simplex gradient is the plane through the
    # opposite side and time divided by twice the spacetime prism swept per unit time.
    gradients = (time & sides) / (2 * (time & patch))                        # [3] Plane+[1]
    face_potentials = triangle.incidence[2, 0] * corner_potentials           # [3] Plane+[1]
    face_field = (gradients ^ face_potentials).sum(axis=0)                   # Field+[1]
    # Each corner advances through spacetime along its world line `time + velocity` per unit time.
    corner_motions = time + corner_velocities                                # Point-[3]
    interior_readings = corner_potentials & corner_motions                   # Scalar+[3]
    endpoint_differences = triangle.boundary[2, 1] * (triangle.d[0] * interior_readings)  # [3] Scalar+[1]
    edge_motions = triangle.incidence[2, 1] * (triangle.incidence[1, 0].mean(axis=0) * corner_motions)  # [3] Point-[1]
    world_sheets = edge_motions & sides                                      # [3] Patch-[1]
    # Pairing the face field with each edge's world sheet gives the magnetic flux crossing the moving edge.
    swept_fluxes = face_field & world_sheets                                 # [3] Scalar+[1]
    return endpoint_differences, swept_fluxes, world_sheets, face_field


endpoint_differences, swept_fluxes, _, face_field = cartan_decomposition(
    triangle_corners, corner_potentials, corner_velocities,
)
save_figure(
    render.cartan_ribbon_figure(triangle_corners, corner_potentials, corner_velocities,
                                ribbon_step, endpoint_differences, swept_fluxes),
    "lie_advection_cartan",
);

Orange arrows carry the three corners to the dashed displaced triangle, sweeping shaded ribbons coloured by their world-sheet magnetic flux while blue streaks inside the initial triangle show the potential's direction and strength. In the bar chart on the right, the blue endpoint differences cancel in the loop sum, leaving the dark total circulation rate equal to the orange swept-flux sum.

In differential-form notation, the rate of a 1-form integrated along a moving edge $e(t)$ splits via Reynolds transport and Cartan's formula into $\frac{d}{dt}\int_{e(t)} A = \int_{\partial e} \iota_v A + \int_e \iota_v(dA)$.

In [ ]:
print("Endpoint differences:", endpoint_differences.sites.sum().to_array())
print("World-sheet fluxes:", swept_fluxes.sites.sum().to_array())

## Magnetic flux advection on a swirling disk

Across a triangulated disk, a shearing vortex follows the level curves of a streamfunction whose value and slope vanish on the rim so the boundary fluid stays at rest. Neighbouring triangles share each interior edge's swept spatial patch, so magnetic flux leaving one triangle enters its neighbour while differential rotation stretches two opposite-sign lobes into thin spiral layers. Executing the time integration and rendering the ninety-frame animation below takes about ten seconds.

In [ ]:
from examples.electromagnetism.lie_advection.scenarios import (
    disk_radius, dt, ellipticity, frame_duration_ms, lobe_centres,
    lobe_radius, lobe_strengths, n_cases, n_disk_divisions, n_frames,
    n_steps, n_steps_per_frame, resistivities, swirl_rate,
)

In [ ]:
def vortex_streamfunction(vertices: Point, swirl_rate: float, ellipticity: float) -> Scalar:
    """An elliptical shearing vortex has streamfunction (`Scalar+[V]`) vanishing smoothly in value and slope on the unit circle."""
    abscissa, ordinate = mv.x & vertices, mv.y & vertices                    # Scalar+[V], Scalar+[V]
    radial_envelope = (1 - vertices.squared()).squared()                     # Scalar+[V]
    quadrupole = 1 + ellipticity * (abscissa.squared() - ordinate.squared()) # Scalar+[V]
    return swirl_rate * radial_envelope * quadrupole                         # Scalar+[V]


def initial_cochain(topology: Topology, vertices: Point, lobe_centres: Point,
                    lobe_strengths: np.ndarray, lobe_radius: float) -> Scalar:
    """Localized Gaussian magnetic potentials around `lobe_centres` pair with oriented edges to form the initial circulation cochain (`Scalar+[E]`)."""
    edges = topology.d[0] * vertices                                         # Point-[E]
    edge_midpoints = topology.incidence[1, 0].mean(axis=0) * vertices        # Point-[E]
    displacements = edge_midpoints - lobe_centres                            # [...] Point-[E]
    # Pairing the spatial bivector with the displacement from each lobe centre forms a circulating potential plane.
    envelopes = (-displacements.squared() / (2 * lobe_radius**2)).exp()      # [...] Scalar+[E]
    potential = ((envelopes * lobe_strengths) * (mv.xy & displacements)).sum(axis=0)  # Plane+[E]
    return potential & edges                                                 # Scalar+[E]


topology, coordinates = shapes.hexagonal_disk(disk_radius, n_disk_divisions)
vertices = mv(Upright, coordinates).dual().field()                           # Point-[V]
stream = vortex_streamfunction(vertices, swirl_rate, ellipticity)            # Scalar+[V]
start_cochain = initial_cochain(topology, vertices, lobe_centres, lobe_strengths, lobe_radius)  # Scalar+[E]

In [ ]:
def whitney_curl(topology: Topology, vertices: Point) -> CurlMap:
    """Corner simplex gradients wedge in pairs across oriented edges to form the piecewise-constant Whitney magnetic 2-form curl (`(Magnetic+ <- Scalar+)[F <- E]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    prisms = time & ((sides[0] & sides[1]) / 2)                              # Scalar-[F]
    corner_gradients = (time & sides) / (2 * prisms)                         # [3] Plane+[F]
    # Each edge's Whitney 1-form has exterior derivative equal to twice the wedge of its endpoint simplex gradients.
    tails, heads = topology.corners[2, 1]
    curls = 2 * (corner_gradients[tails] ^ corner_gradients[heads])          # [3] Magnetic+[F]
    return (spdiag(curls * Scalar) * topology.oriented[2, 1]).sum(axis=0)    # (Magnetic+ <- Scalar+)[F <- E]


def swept_edge_patches(topology: Topology, vertices: Point, stream: Scalar) -> Patch:
    """Fixed and comoving edge world sheets under `stream` differ by the swept spatial patches (`Patch-[E]`)."""
    edges = topology.d[0] * vertices                                         # Point-[E]
    sides = topology.boundary[2, 1] * edges                                  # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    prisms = time & patches                                                  # Scalar-[F]
    areas = prisms.norm()                                                    # Scalar+[F]
    normals = patches / areas                                                # Patch-[F]
    corner_gradients = (time & sides) / (2 * prisms)                         # [3] Plane+[F]
    gradient = (spdiag(corner_gradients * Scalar) * topology.incidence[2, 0]).sum(axis=0)  # (Plane+ <- Scalar+)[F <- V]
    face_velocity = gradient(stream) ^ normals                               # Point-[F]
    face_to_edge = topology.incidence[2, 1].sum(axis=0).adjugate()           # Scalar+[E <- F]
    edge_counts = face_to_edge * (areas / areas)                             # Scalar+[E]
    to_edges = spdiag(1 / edge_counts) * face_to_edge                        # (Scalar+ <- Scalar+)[E <- F]
    edge_velocity = (to_edges * Point)(face_velocity)                        # Point-[E]
    fixed_sheets = time & edges                                              # Patch-[E]
    comoving_sheets = (time + edge_velocity) & edges                         # Patch-[E]
    return fixed_sheets - comoving_sheets                                    # Patch-[E]


curl = whitney_curl(topology, vertices)                                      # (Magnetic+ <- Scalar+)[F <- E]
swept_patches = swept_edge_patches(topology, vertices, stream)               # Patch-[E]

In [ ]:
def advection_map(topology: Topology, vertices: Point, curl: CurlMap,
                  swept_patches: Patch) -> ScalarMap:
    """Adjacent face magnetic fields average onto edges and pair with `swept_patches` to form the Lie advection map (`(Scalar+ <- Scalar+)[E <- E]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    areas = (time & ((sides[0] & sides[1]) / 2)).norm()                      # Scalar+[F]
    # Averaging adjacent face fields onto each edge preserves uniform fields and skew-symmetry in magnetic energy.
    face_to_edge = topology.incidence[2, 1].sum(axis=0).adjugate()           # Scalar+[E <- F]
    edge_counts = face_to_edge * (areas / areas)                             # Scalar+[E]
    to_edges = spdiag(1 / edge_counts) * face_to_edge                        # (Scalar+ <- Scalar+)[E <- F]
    edge_curl = (to_edges * Magnetic)(curl)                                  # (Magnetic+ <- Scalar+)[E <- E]
    return (Magnetic & swept_patches)(edge_curl)                             # (Scalar+ <- Scalar+)[E <- E]


def curl_stiffness_map(topology: Topology, vertices: Point, curl: CurlMap) -> StiffnessMap:
    """Pulling the area-weighted vacuum duality map back through `curl` forms the magnetic stiffness (`(Antiscalar- <- Scalar+)[E <- E]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    areas = (time & ((sides[0] & sides[1]) / 2)).norm()                      # Scalar+[F]
    vacuum = areas * Magnetic.dual()                                         # (Patch- <- Magnetic+)[F]
    return curl.adjugate()(vacuum(curl))                                     # (Antiscalar- <- Scalar+)[E <- E]


advection = advection_map(topology, vertices, curl, swept_patches)           # (Scalar+ <- Scalar+)[E <- E]
curl_stiffness = curl_stiffness_map(topology, vertices, curl)                # (Antiscalar- <- Scalar+)[E <- E]

In [ ]:
def curl_laplacian_map(topology: Topology, curl_stiffness: StiffnessMap) -> ScalarMap:
    """Restricting unit-resistance magnetic diffusion to interior edges keeps boundary circulation fixed (`(Scalar+ <- Scalar+)[E <- E]`)."""
    interior = topology.reading(topology.interior[1], 1).adjugate() * Scalar # (Scalar+ <- Scalar+)[E <- E_interior]
    unit_resistance = Antiscalar.dual_inverse()                              # [] Scalar+ <- Antiscalar-
    return interior(unit_resistance(interior.adjugate()(curl_stiffness)))    # (Scalar+ <- Scalar+)[E <- E]


def flux_density_map(topology: Topology, vertices: Point) -> ScalarMap:
    """Dividing face boundary sums by triangle areas maps edge cochains to face magnetic flux densities (`(Scalar+ <- Scalar+)[F <- E]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    areas = (time & ((sides[0] & sides[1]) / 2)).norm()                      # Scalar+[F]
    return spdiag(1 / areas) * (topology.d[1] * Scalar)                      # (Scalar+ <- Scalar+)[F <- E]


curl_laplacian = curl_laplacian_map(topology, curl_stiffness)                # (Scalar+ <- Scalar+)[E <- E]
flux_density = flux_density_map(topology, vertices)                          # (Scalar+ <- Scalar+)[F <- E]
rate = advection - curl_laplacian * resistivities[:, None]                   # [n_cases] (Scalar+ <- Scalar+)[E <- E]
batched_start = stack((start_cochain, start_cochain))                        # [n_cases] Scalar+[E]

In [ ]:
def cochain_trajectory(initial: Scalar, rate: ScalarMap, dt: float, n_steps: int) -> Iterator[Scalar]:
    """Yields edge circulation cochains (`[...] Scalar+[E]`) across `n_steps` fourth-order Runge-Kutta steps."""
    cochain = initial                                                        # [...] Scalar+[E]
    yield cochain
    for _ in range(n_steps):
        first_rate = rate(cochain)                                           # [...] Scalar+[E]
        second_rate = rate(cochain + (dt / 2) * first_rate)                  # [...] Scalar+[E]
        third_rate = rate(cochain + (dt / 2) * second_rate)                  # [...] Scalar+[E]
        fourth_rate = rate(cochain + dt * third_rate)                        # [...] Scalar+[E]
        cochain = cochain + (dt / 6) * (first_rate + 2 * second_rate + 2 * third_rate + fourth_rate)  # [...] Scalar+[E]
        yield cochain


sampled_cochains = show_states(
    islice(cochain_trajectory(batched_start, rate, dt, n_steps), 0, None, n_steps_per_frame),
    render.advection_frames, topology, vertices, flux_density,
    name="lie_advection_flow", duration_ms=frame_duration_ms,
)

Orange and blue shade positive and negative magnetic flux density at a fixed colour scale, with dark contours marking equal flux levels. Differential rotation winds the ideal conductor on the left into tight alternating filaments, whereas interior resistance in the conductor on the right broadens the filaments and diminishes their peak amplitude.

## Magnetic energy conservation and resistive dissipation

Reading the streamed edge cochains at the end of the run compares the final flux distributions against the time histories of magnetic energy and total flux. Because averaging face fields onto edges makes the ideal advection operator skew-adjoint with respect to the magnetic stiffness pairing, ideal advection conserves magnetic energy while interior resistance dissipates energy monotonically without changing boundary circulation or total disk flux.

In [ ]:
cochain_states = stack(sampled_cochains)                                     # [n_frames, n_cases] Scalar+[E]
d1 = topology.d[1] * Scalar                                                  # (Scalar+ <- Scalar+)[F <- E]
energies = (cochain_states & curl_stiffness(cochain_states)).sites.sum() / 2 # [n_frames, n_cases] Scalar+
total_fluxes = d1(cochain_states).sites.sum()                                # [n_frames, n_cases] Scalar+
final_flux_density = flux_density(cochain_states[-1])                        # [n_cases] Scalar+[F]
frame_times = np.arange(n_frames) * (n_steps_per_frame * dt)

save_figure(
    render.advection_comparison_figure(topology, vertices, final_flux_density,
                                       frame_times, energies, total_fluxes),
    "lie_advection_snapshots",
);

The left and middle disks show the final ideal and resistive magnetic flux densities side by side. On the right, relative to their initial values, the dark curve tracks the conserved ideal magnetic energy, the orange curve shows resistive magnetic energy decay, and the dashed teal curve confirms that total magnetic flux stays constant under resistive diffusion.

In [ ]:
print("Relative final magnetic energy (ideal, resistive):", (energies[-1] / energies[0]).to_array())

In [ ]:
# checks
# Associativity of the regressive product equates world-sheet flux to the interior-product line integral,
# and endpoint differences cancel around the closed triangle boundary.
triangle_sides = triangle.boundary[2, 1] * (triangle.d[0] * triangle_corners)
edge_motions = triangle.incidence[2, 1] * (triangle.incidence[1, 0].mean(axis=0) * (time + corner_velocities))
interior_line_integrals = (face_field & edge_motions) & triangle_sides
assert ((swept_fluxes - interior_line_integrals).abs() < 1e-12).all()
assert (endpoint_differences.sum(axis=0).abs() < 1e-12).all()

# Across every edge, the swept spatial patch area equals minus the streamfunction difference.
d0 = topology.d[0] * Scalar
assert (((mv.xy & swept_patches) + d0(stream)).abs() < 1e-12).all()

# Ideal advection produces zero instantaneous magnetic power and conserves flux and energy,
# while interior resistance dissipates magnetic energy monotonically.
ideal_initial_rate = advection(start_cochain)
assert ((ideal_initial_rate & curl_stiffness(start_cochain)).sites.sum().abs() < 1e-12).all()
assert ((total_fluxes - total_fluxes[0]).abs() < 1e-12).all()
assert (((energies[:, 0] / energies[0, 0]) - 1).abs() < 1e-4).all()
assert ((energies[1:, 1] - energies[:-1, 1]) < 0).all()